# 6.38 - CertCF Epsilon Volume Ablation

Diagnostic COMPAS notebook for testing whether larger center-certified radii produce useful certified regions. We compare `shrink + binary search` against `shrink + fitted alpha rule`, using model-predicted labels everywhere.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from certcf import NearestOppositeClassClearanceStrategy
from counterfactuals.datasets.loaders import CompasDataset
from counterfactuals.methods.certcf import CertCF
from scripts.benchmark import _build_torch_model_from_checkpoint

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 120})

DATASET = 'compas'
SEED = 42
N_SUPPORT = 500
NORM = 1
CLASSIFICATION_MARGIN = 1.0e-4
BINARY_SEARCH_STEPS = 10
N_VOLUME_SAMPLES = 2048
VOLUME_TOL = 1.0e-8
DEVICE = 'auto'
CHECKPOINT = ROOT / 'checkpoints/compas_classifier/best.ckpt'

rng = np.random.default_rng(SEED)
volume_rng = np.random.default_rng(SEED + 137)
print({'dataset': DATASET, 'n_support': N_SUPPORT, 'n_volume_samples': N_VOLUME_SAMPLES})


## Load COMPAS And Sample Support Points

In [ ]:
def stratified_indices(labels: np.ndarray, n_total: int, rng: np.random.Generator) -> np.ndarray:
    labels = np.asarray(labels, dtype=np.int64)
    classes = np.unique(labels)
    base = int(n_total) // len(classes)
    remainder = int(n_total) - base * len(classes)
    chosen = []
    for pos, cls in enumerate(classes):
        idx = np.flatnonzero(labels == cls)
        take = min(len(idx), base + int(pos < remainder))
        chosen.append(rng.choice(idx, size=take, replace=False))
    out = np.concatenate(chosen)
    rng.shuffle(out)
    return out


dataset = CompasDataset(data_dir=str(ROOT / 'data'), seed=SEED)
dataset.load()
x_train_full, y_train = dataset.get_train()
spec = dataset.spec

model = _build_torch_model_from_checkpoint(
    checkpoint=str(CHECKPOINT),
    device=DEVICE,
    dataset_module=DATASET,
    hidden_dims=[64, 32],
    dropout=0.2,
)

y_train_pred = model.predict(x_train_full).astype(np.int64)
support_idx = stratified_indices(y_train_pred, N_SUPPORT, rng)
x_support = x_train_full[support_idx]
y_support = y_train_pred[support_idx]

LOAD_SUMMARY_DF = pd.DataFrame([{
    'dataset': DATASET,
    'encoded_dim': x_train_full.shape[1],
    'train_rows': len(x_train_full),
    'support_rows': len(x_support),
    'train_label_agreement': float(np.mean(y_train_pred == y_train)),
}])
display(LOAD_SUMMARY_DF.round(4))
print('support counts by model prediction:', dict(zip(*np.unique(y_support, return_counts=True))))


## Build CertCF Atlases

In [ ]:
def collect_epsilon_diagnostics(method: CertCF, variant: str) -> pd.DataFrame:
    rows = []
    atlas = method.atlas
    for label in atlas.class_labels:
        bd = atlas.bounds[int(label)]
        eps_initial = np.asarray(bd['eps_initial'], dtype=float)
        eps_final = np.asarray(bd['eps'], dtype=float)
        certified = np.asarray(bd['adaptive_eps_center_certified'], dtype=bool)
        shrinks = np.asarray(bd['adaptive_eps_n_shrinks'], dtype=float)
        binary_steps = np.asarray(bd['adaptive_eps_n_binary_steps'], dtype=float)
        slack = np.asarray(bd['adaptive_eps_center_slack'], dtype=float)
        alpha_final = np.divide(eps_final, eps_initial, out=np.full_like(eps_final, np.nan), where=eps_initial > 0)
        rows.append(pd.DataFrame({
            'variant': variant,
            'class_label': int(label),
            'polytope_idx': np.arange(len(eps_final), dtype=int),
            'clearance': eps_initial,
            'eps_final': eps_final,
            'alpha_final': alpha_final,
            'center_certified': certified,
            'n_shrinks': shrinks,
            'n_binary_steps': binary_steps,
            'center_slack': slack,
        }))
    return pd.concat(rows, ignore_index=True)


def build_certcf(alpha: float, binary_steps: int) -> CertCF:
    return CertCF(
        model=model,
        norm=NORM,
        distance_norm=NORM,
        lirpa_method='backward',
        eps_strategy=NearestOppositeClassClearanceStrategy(alpha=float(alpha)),
        batch_size=128,
        ohe_slices=list(spec.categorical_slices),
        default_query_method='nearest_anchor',
        query_k_candidates=5,
        k_per_class=None,
        classification_margin=CLASSIFICATION_MARGIN,
        random_seed=SEED,
        adaptive_eps=True,
        adaptive_eps_shrink_factor=0.5,
        adaptive_eps_max_shrinks=8,
        adaptive_eps_min=1.0e-6,
        adaptive_eps_center_tol=1.0e-6,
        adaptive_eps_binary_search_steps=int(binary_steps),
    )


def fit_alpha_hat(epsilon_df: pd.DataFrame) -> float:
    fit_df = epsilon_df[
        epsilon_df['center_certified']
        & np.isfinite(epsilon_df['clearance'])
        & np.isfinite(epsilon_df['eps_final'])
        & (epsilon_df['clearance'] > 0)
    ]
    x = fit_df['clearance'].to_numpy(dtype=float)
    y = fit_df['eps_final'].to_numpy(dtype=float)
    return float(np.dot(x, y) / np.dot(x, x))


build_rows = []

binary_variant = f'shrink + binary search ({BINARY_SEARCH_STEPS} steps)'
binary_method = build_certcf(alpha=1.0, binary_steps=BINARY_SEARCH_STEPS)
t0 = time.perf_counter()
binary_method.fit(x_train=x_support, y_train=y_support)
binary_build_time_s = time.perf_counter() - t0
BINARY_EPSILON_DF = collect_epsilon_diagnostics(binary_method, binary_variant)
alpha_hat = fit_alpha_hat(BINARY_EPSILON_DF)

alpha_variant = f'shrink + alpha rule (alpha={alpha_hat:.3f})'
alpha_rule_method = build_certcf(alpha=alpha_hat, binary_steps=0)
t0 = time.perf_counter()
alpha_rule_method.fit(x_train=x_support, y_train=y_support)
alpha_build_time_s = time.perf_counter() - t0
ALPHA_RULE_EPSILON_DF = collect_epsilon_diagnostics(alpha_rule_method, alpha_variant)

EPSILON_DF = pd.concat([BINARY_EPSILON_DF, ALPHA_RULE_EPSILON_DF], ignore_index=True)

BUILD_SUMMARY_DF = pd.DataFrame([
    {
        'variant': binary_variant,
        'build_time_s': binary_build_time_s,
        'mean_lirpa_calls': float((1 + BINARY_EPSILON_DF['n_shrinks'] + BINARY_EPSILON_DF['n_binary_steps']).mean()),
        'center_certified_pct': 100.0 * float(BINARY_EPSILON_DF['center_certified'].mean()),
        'median_final_epsilon': float(BINARY_EPSILON_DF['eps_final'].median()),
        'median_alpha_final': float(BINARY_EPSILON_DF['alpha_final'].median()),
    },
    {
        'variant': alpha_variant,
        'build_time_s': alpha_build_time_s,
        'mean_lirpa_calls': float((1 + ALPHA_RULE_EPSILON_DF['n_shrinks'] + ALPHA_RULE_EPSILON_DF['n_binary_steps']).mean()),
        'center_certified_pct': 100.0 * float(ALPHA_RULE_EPSILON_DF['center_certified'].mean()),
        'median_final_epsilon': float(ALPHA_RULE_EPSILON_DF['eps_final'].median()),
        'median_alpha_final': float(ALPHA_RULE_EPSILON_DF['alpha_final'].median()),
    },
])

display(pd.DataFrame([{'fitted_alpha_hat': alpha_hat}]).round(4))
display(BUILD_SUMMARY_DF.round(4))


## Estimate Retained Certified Volume

In [ ]:
def sample_uniform_l1_ball(center: np.ndarray, eps: float, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    center = np.asarray(center, dtype=np.float64)
    d = center.size
    expo = rng.exponential(scale=1.0, size=(n_samples, d + 1))
    magnitudes = expo[:, :d] / expo.sum(axis=1, keepdims=True)
    signs = rng.choice(np.array([-1.0, 1.0]), size=(n_samples, d))
    return center[None, :] + float(eps) * signs * magnitudes


def log_l1_ball_volume(eps: float, dim: int) -> float:
    if eps <= 0:
        return -np.inf
    return dim * math.log(2.0) + dim * math.log(float(eps)) - math.lgamma(dim + 1)


def estimate_volume_diagnostics(method: CertCF, variant: str, n_samples: int = N_VOLUME_SAMPLES) -> pd.DataFrame:
    rows = []
    atlas = method.atlas
    for label in atlas.class_labels:
        bd = atlas.bounds[int(label)]
        centers = np.asarray(bd['X'], dtype=np.float64)
        eps_final = np.asarray(bd['eps'], dtype=np.float64)
        eps_initial = np.asarray(bd.get('eps_initial', eps_final), dtype=np.float64)
        certified = np.asarray(bd.get('adaptive_eps_center_certified', np.ones(len(centers), dtype=bool)), dtype=bool)
        lA = np.asarray(bd['lA'], dtype=np.float64)
        lbias = np.asarray(bd['lbias'], dtype=np.float64)
        dim = centers.shape[1]

        for idx, center in enumerate(centers):
            eps = float(eps_final[idx])
            samples = sample_uniform_l1_ball(center, eps, n_samples, volume_rng)
            A = lA[idx].reshape(-1, dim)
            b = lbias[idx].reshape(-1)
            if len(b):
                slack = samples @ A.T + b[None, :] - CLASSIFICATION_MARGIN
                inside = np.all(slack >= -VOLUME_TOL, axis=1)
            else:
                inside = np.ones(n_samples, dtype=bool)
            retained = float(inside.mean())
            retained_for_log = max(retained, 0.5 / n_samples)
            log_ball = log_l1_ball_volume(eps, dim)
            rows.append({
                'variant': variant,
                'class_label': int(label),
                'polytope_idx': int(idx),
                'dim': int(dim),
                'eps_final': eps,
                'clearance': float(eps_initial[idx]),
                'alpha_final': float(eps / eps_initial[idx]) if eps_initial[idx] > 0 else np.nan,
                'center_certified': bool(certified[idx]),
                'retained_fraction': retained,
                'zero_hit': bool(retained == 0.0),
                'log_l1_ball_volume': log_ball,
                'log_estimated_polytope_volume': log_ball + math.log(retained_for_log),
            })
    return pd.DataFrame(rows)


BINARY_VOLUME_DF = estimate_volume_diagnostics(binary_method, binary_variant)
ALPHA_RULE_VOLUME_DF = estimate_volume_diagnostics(alpha_rule_method, alpha_variant)
VOLUME_DF = pd.concat([BINARY_VOLUME_DF, ALPHA_RULE_VOLUME_DF], ignore_index=True)

assert VOLUME_DF['retained_fraction'].between(0.0, 1.0).all()
assert np.isfinite(VOLUME_DF['log_l1_ball_volume']).all()

display(VOLUME_DF.head().round(4))


## Summary Tables

In [ ]:
VOLUME_SUMMARY_DF = (
    VOLUME_DF
    .groupby('variant', observed=True)
    .agg(
        n_polytopes=('retained_fraction', 'size'),
        eps_median=('eps_final', 'median'),
        retained_mean=('retained_fraction', 'mean'),
        retained_median=('retained_fraction', 'median'),
        retained_q25=('retained_fraction', lambda s: float(np.quantile(s, 0.25))),
        retained_q75=('retained_fraction', lambda s: float(np.quantile(s, 0.75))),
        zero_hit_pct=('zero_hit', lambda s: 100.0 * float(np.mean(s))),
        log_l1_ball_volume_median=('log_l1_ball_volume', 'median'),
        log_estimated_polytope_volume_median=('log_estimated_polytope_volume', 'median'),
    )
    .reset_index()
)

ABLATION_SUMMARY_DF = BUILD_SUMMARY_DF.merge(
    VOLUME_SUMMARY_DF,
    on='variant',
    how='left',
)
display(ABLATION_SUMMARY_DF.round(4))


## Plots

In [ ]:
colors = {
    binary_variant: '#0072B2',
    alpha_variant: '#D55E00',
}

fig, ax = plt.subplots(figsize=(6.2, 4.2))
for variant, sub in VOLUME_DF.groupby('variant', observed=True):
    ax.scatter(
        sub['eps_final'],
        sub['retained_fraction'],
        s=14,
        alpha=0.50,
        color=colors.get(variant),
        label=variant,
    )
ax.set_xscale('log')
ax.set_xlabel('final epsilon')
ax.set_ylabel('retained fraction')
ax.set_title('Certified fraction inside final L1 ball')
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
for variant, sub in VOLUME_DF.groupby('variant', observed=True):
    color = colors.get(variant)
    axes[0].hist(sub['retained_fraction'], bins=30, density=True, alpha=0.42, color=color, label=variant)
    axes[1].hist(sub['log_estimated_polytope_volume'], bins=30, density=True, alpha=0.42, color=color, label=variant)

axes[0].set_xlabel('retained fraction')
axes[0].set_ylabel('density')
axes[0].set_title('Retained certified fraction')
axes[1].set_xlabel('log estimated certified volume')
axes[1].set_title('Estimated certified volume')
for ax in axes:
    ax.grid(True, alpha=0.25)
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()
